# Multi-Model AI Technical Tutor

This project uses two different LLMs together to answer technical
questions.

1. Llama 3.2 creates an initial explanation.
2. GPT reviews the explanation for accuracy and clarity.
3. Llama receives GPT's feedback and creates an improved final answer.

The goal is to demonstrate how multiple LLM calls can be orchestrated
together in a simple Python workflow.

In [ ]:
import os
import requests

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display


In [ ]:
# constants

MODEL_GPT = 'gpt-5.4-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No OpenAI API key found.")
elif not api_key.startswith("sk-"):
    print("An API key was found, but it doesn't look like an OpenAI key.")
else:
    print("OpenAI API key found.")

In [ ]:
openai = OpenAI(api_key=api_key)

In [ ]:
response = requests.get("http://localhost:11434")

print(response.text)

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

# Testing both connections

In [ ]:
response = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        {"role": "system", "content": "You are a real outback Australian that knows how to greet people in a friendly way"},
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)

print(response.choices[0].message.content)

In [ ]:
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role": "system", "content": "You are a mobster that talks like they are from the 1920's"},
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)

print(response.choices[0].message.content)

# Ollama as the Teacher
- Define the question
- Create the llama system prompt
- Create ask_llama function that takes in a question variable
- Store ask_llama function in variable (for input later) and call function

In [ ]:
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
llama_teacher_prompt = """
You are a patient technical tutor teaching a student who is learning
Python and AI engineering.

Explain technical concepts clearly and accurately.

When appropriate:
- explain the concept in simple language
- break complicated ideas into steps
- provide an analogy
- provide a small example
- avoid unnecessary jargon

Your goal is understanding, not simply providing an answer.
"""

In [ ]:
def ask_llama(question):

    messages = [
        {
            "role": "system",
            "content": llama_teacher_prompt
        },
        {
            "role": "user",
            "content": question
        }
    ]

    response = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages
    )
    
    return response.choices[0].message.content

In [ ]:
llama_answer = ask_llama(question)

display(Markdown(llama_answer))

# GPT as the Reviewer

In [ ]:
gpt_reviewer_prompt = """
You are a senior technical educator reviewing an explanation written
by another AI tutor.

Evaluate the explanation for:

1. Technical accuracy
2. Clarity for a beginner
3. Missing important information
4. Unnecessary jargon
5. Quality of examples or analogies

Be constructive.

Do not rewrite the entire answer yet.

Instead, provide specific feedback that another tutor can use to
improve the explanation.
"""

In [ ]:
def review_with_gpt(question, llama_answer):

    review_request = f"""
Original student question:

{question}

Here is the explanation provided by the first tutor:

{llama_answer}

Please review this explanation and provide constructive feedback.
"""

    messages = [
        {
            "role": "system",
            "content": gpt_reviewer_prompt
        },
        {
            "role": "user",
            "content": review_request
        }
    ]

    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
gpt_review = review_with_gpt(question, llama_answer)

display(Markdown(gpt_review))

# Feedback to Llama 3.2

In [ ]:
llama_revision_prompt = """
You are a patient technical tutor.

You previously produced an explanation for a student.
A second AI educator has reviewed your explanation.

Use the review to create a better final explanation.

The final answer should:

- directly answer the student's question
- be technically accurate
- be understandable to a beginner
- explain important terminology
- include an example when helpful
- incorporate useful reviewer feedback
- not mention the review process or the other AI

Return only the improved explanation.
"""

In [ ]:
def improve_with_llama(question, llama_answer, gpt_review):

    revision_request = f"""
Original student question:

{question}

Your original explanation:

{llama_answer}

Reviewer's feedback:

{gpt_review}

Now produce the improved final explanation.
"""

    messages = [
        {
            "role": "system",
            "content": llama_revision_prompt
        },
        {
            "role": "user",
            "content": revision_request
        }
    ]

    response = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages
    )
    return response.choices[0].message.content

In [ ]:
final_answer = improve_with_llama(question, llama_answer, gpt_review)

display(Markdown(final_answer))

# One Function to Rule Them All

In [ ]:
def technical_tutor(question):

    print("STEP 1: Llama is creating an initial explanation...\n")
    llama_answer = ask_llama(question)

    print("STEP 2: GPT is reviewing Llama's explanation...\n")
    gpt_review = review_with_gpt(
        question,
        llama_answer
    )

    print("STEP 3: Llama is revising the explanation...\n")
    final_answer = improve_with_llama(
        question,
        llama_answer,
        gpt_review
    )

    print("Workflow complete.")

    return {
        "initial_answer": llama_answer,
        "review": gpt_review,
        "final_answer": final_answer
    }

# Test Orchestration Layer

In [ ]:
result = technical_tutor(question)

display(Markdown("## Llama's Initial Answer"))
display(Markdown(result["initial_answer"]))

display(Markdown("## GPT's Review"))
display(Markdown(result["review"]))

display(Markdown("## Final Improved Answer"))
display(Markdown(result["final_answer"]))

# Additional Tests

In [ ]:
question = """
What is the difference between an API, an endpoint,
and a Python client library?
"""

In [ ]:
result = technical_tutor(question)

display(Markdown(result["final_answer"]))

In [ ]:
question = """
Explain why LLM API calls are stateless and how a chatbot
can appear to remember a conversation.
"""

In [ ]:
result = technical_tutor(question)

display(Markdown(result["final_answer"]))

# What I Learned

This project helped me understand that separate LLM calls can be
combined into a larger workflow.

Each LLM call is stateless. Llama does not directly communicate
with GPT. Instead, my Python program stores the output from one model
in a variable and includes it in the prompt sent to another model.

I assigned different responsibilities to the models:

- Llama acts as the primary technical tutor.
- GPT acts as a reviewer.
- Llama uses the review to improve its original response.

The technical_tutor() function acts as a simple orchestration layer
that coordinates these separate model calls.

This made it clearer to me how more complicated AI systems can be
constructed from multiple specialized LLM calls.